# CMPE 249 detection: Stage 2 runner
Runtime → Change runtime type → GPU (pick one GPU type and use it for every latency number).
Data goes to Colab local disk (`/content/data`); results go to Drive so a disconnect loses nothing.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/YOUR_GITHUB_USER/cmpe249-detection.git'  # <-- edit
!git clone {REPO_URL} /content/cmpe249-detection
%cd /content/cmpe249-detection
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
os.environ['CMPE249_DATA'] = '/content/data'                                   # local disk: fast reads
os.environ['CMPE249_WEIGHTS'] = '/content/weights'
os.environ['CMPE249_RESULTS'] = '/content/drive/MyDrive/cmpe249-hw/results'    # Drive: survives disconnects
os.makedirs(os.environ['CMPE249_RESULTS'], exist_ok=True)
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# Data prep (once per Colab session; local disk is wiped on disconnect)
!python -m src.data.kitti_to_coco            # full KITTI val (1496 images)
!python -m src.data.coco_subset --full       # full COCO val2017 (5000 images, fetched by coco_url)

In [ ]:
# One config end to end: predictions + metrics + run_meta + env logs -> Drive
!python -m src.predict --config configs/full/yolo26s_kitti.yaml
# Other runs: configs/full/{yolo26s,rtdetr_l,frcnn_v2}_{coco,kitti}.yaml
# Latency:    !python -m src.bench_latency --config configs/full/yolo26s_kitti.yaml [--half]